# 07 — Literature Integration

This notebook creates a structured scientific-evidence knowledge base for the pilot-cluster genes selected in Notebook 06. It retrieves and organizes literature, but it does **not** combine the evidence into a biological conclusion. Synthesis is reserved for Notebook 08.

Three evidence layers remain separate throughout:

1. **Dataset observations** — loaded directly from Phase 6.
2. **Literature findings** — attached to verified PubMed records.
3. **Future interpretation** — explicitly deferred.

No disease diagnosis, donor inference, or unsupported speculation is performed.

In [1]:
from collections import Counter
from pathlib import Path
import re
import time
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 120)

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "results" / "phase6").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the PBMC3k project root with Phase 6 outputs.")

PROJECT = find_project_root(Path.cwd().resolve())
PHASE6 = PROJECT / "results" / "phase6"
PHASE7 = PROJECT / "results" / "phase7"
GENE_DIR = PHASE7 / "genes"
PHASE7.mkdir(parents=True, exist_ok=True)
GENE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT}")
print(f"Phase 7 output: {PHASE7}")

Project root: /Users/shelyjain/Desktop/Desktop - Shely's Macbook Pro/cosmos/26-the-backpropagators-analysis/PBMC3k
Phase 7 output: /Users/shelyjain/Desktop/Desktop - Shely's Macbook Pro/cosmos/26-the-backpropagators-analysis/PBMC3k/results/phase7


## 1. Load Phase 6 and detect the pilot cluster

The pilot cluster is read from the generated report heading rather than hard-coded. Only selected genes belonging to that cluster are retained.

In [2]:
ranked_path = PHASE6 / "ranked_marker_genes.csv"
selected_path = PHASE6 / "selected_marker_genes.csv"
pilot_summary_path = PHASE6 / "pilot_cluster_summary.md"
for required in [ranked_path, selected_path, pilot_summary_path]:
    if not required.exists():
        raise FileNotFoundError(f"Required Phase 6 input is missing: {required}")

ranked = pd.read_csv(ranked_path)
selected = pd.read_csv(selected_path)
pilot_summary = pilot_summary_path.read_text(encoding="utf-8")

pilot_match = re.search(r"^# Cluster\s+([^:]+):\s*(.+?)\s*$", pilot_summary, flags=re.MULTILINE)
if not pilot_match:
    raise ValueError("Could not detect the pilot cluster from pilot_cluster_summary.md")
pilot_cluster = pilot_match.group(1).strip()
pilot_cell_type = pilot_match.group(2).strip()
selected["cluster"] = selected["cluster"].astype(str)
pilot_genes = selected[selected["cluster"] == pilot_cluster].sort_values("representative_rank").copy()
if pilot_genes.empty:
    raise ValueError(f"No selected markers were found for detected pilot cluster {pilot_cluster}.")

dataset_columns = [
    "cluster", "cell_type", "representative_rank", "gene", "marker_score",
    "avg_log2FC", "adjusted_p_value", "pct_in", "pct_out", "specificity_delta",
]
display(Markdown(f"**Detected pilot:** cluster {pilot_cluster} — {pilot_cell_type}"))
display(pilot_genes[dataset_columns])

**Detected pilot:** cluster 5 — NK cells

,cluster,cell_type,representative_rank,gene,marker_score,avg_log2FC,adjusted_p_value,pct_in,pct_out,specificity_delta
50,5,NK cells,1,GZMB,142.958910,7.890887,4.623276e-85,0.973856,0.068008,0.905848
51,5,NK cells,2,FGFBP2,115.462342,6.923403,1.184735e-70,0.895425,0.061569,0.833855
52,5,NK cells,3,GNLY,114.826898,7.358587,2.051503e-72,0.915033,0.134809,0.780224
53,5,NK cells,4,PRF1,112.411747,6.527348,9.805125e-83,0.967320,0.106237,0.861083
54,5,NK cells,5,NKG7,104.006494,6.985301,4.393452e-88,1.000000,0.255533,0.744467
55,5,NK cells,6,CST7,90.353588,5.519955,6.498606e-76,0.967320,0.148893,0.818427
56,5,NK cells,7,SPON2,88.511024,6.330124,5.106432e-48,0.738562,0.039437,0.699125
57,5,NK cells,8,GZMA,88.033071,5.479171,2.617524e-74,0.954248,0.150905,0.803343
58,5,NK cells,9,CCL4,72.808254,5.431434,4.898287e-45,0.745098,0.074849,0.670249
59,5,NK cells,10,CTSW,68.444433,4.689354,1.005510e-75,0.986928,0.257143,0.729785


## 2. Search strategy and scientific safeguards

PubMed is the sole metadata authority used here. Candidate papers were screened in the requested priority order: reviews, human evidence, PBMC/blood studies, and single-cell studies. A curated PMID list prevents an automated relevance ranking from silently including papers that merely mention a gene. Metadata are then retrieved from the official NCBI E-utilities service; the notebook fails if a PMID cannot be verified.

The short article notes below are conservative paraphrases of each PubMed abstract. They organize what each publication reports and are not endorsements, replications, or conclusions. Google Scholar was not needed.

In [3]:
GENE_PROFILES = {
    "GZMB": {
        "official_name": "granzyme B",
        "immune_function": "Cytotoxic-granule serine protease discussed in perforin-dependent target-cell killing and in perforin-independent extracellular activity.",
        "immune_cells": "NK cells; cytotoxic CD8 T cells; context-dependent expression in additional immune cells",
        "biological_role": "Effector protease released from cytotoxic granules; reported substrates and effects vary by cellular and disease context.",
        "function_tags": ["cytotoxic granules", "target-cell killing", "serine protease"],
        "pathway_tags": ["perforin–granzyme pathway", "programmed cell death"],
        "disease_tags": ["cancer", "infection", "inflammation"],
        "plain": "GZMB makes granzyme B, one of the protein tools killer immune cells can release when they attack a target cell.",
    },
    "FGFBP2": {
        "official_name": "fibroblast growth factor binding protein 2",
        "immune_function": "Frequently used in single-cell studies as an expression marker of mature cytotoxic NK-cell states; a direct immune mechanism is less well established.",
        "immune_cells": "Cytotoxic NK cells; some cytotoxic T/NKT-cell states",
        "biological_role": "Literature in this collection primarily treats FGFBP2 as a cell-state marker rather than demonstrating a causal NK-cell mechanism.",
        "function_tags": ["cytotoxic cell-state marker", "NK-cell heterogeneity"],
        "pathway_tags": ["cytotoxic lymphocyte program"],
        "disease_tags": ["melanoma", "hepatocellular carcinoma"],
        "plain": "FGFBP2 is often used as a label for a strongly cytotoxic group of NK cells, but its exact immune job is still less clear.",
    },
    "GNLY": {
        "official_name": "granulysin",
        "immune_function": "Cytotoxic and antimicrobial granule protein reported in human NK cells and cytotoxic T cells.",
        "immune_cells": "NK cells; cytotoxic CD8 T cells; gamma-delta T cells",
        "biological_role": "Reported to damage microbial or target-cell membranes, with precursor and mature forms stored in distinct effector-vesicle contexts.",
        "function_tags": ["cytotoxic granules", "antimicrobial defense", "target-cell killing"],
        "pathway_tags": ["cytotoxic lymphocyte degranulation", "host defense"],
        "disease_tags": ["bacterial infection", "pregnancy infection defense"],
        "plain": "GNLY makes granulysin, a protein that some killer immune cells use against microbes and other target cells.",
    },
    "PRF1": {
        "official_name": "perforin 1",
        "immune_function": "Pore-forming cytotoxic-granule protein required for normal killing by NK cells and cytotoxic T cells.",
        "immune_cells": "NK cells; cytotoxic CD8 T cells",
        "biological_role": "Enables cytotoxic granule contents to act on target cells; pathogenic variants can impair lymphocyte cytotoxicity.",
        "function_tags": ["cytotoxic granules", "pore formation", "target-cell killing"],
        "pathway_tags": ["perforin–granzyme pathway", "lymphocyte cytotoxicity"],
        "disease_tags": ["familial hemophagocytic lymphohistiocytosis"],
        "plain": "PRF1 makes perforin, which helps killer immune cells deliver their attack proteins into a target cell.",
    },
    "NKG7": {
        "official_name": "natural killer cell granule protein 7",
        "immune_function": "Granule-associated protein studied as a regulator of cytotoxic granule release in NK and T cells.",
        "immune_cells": "NK cells; cytotoxic CD8 T cells; selected activated CD4 T-cell states",
        "biological_role": "Experimental studies report effects on granule exocytosis, target-cell killing, and downstream inflammation.",
        "function_tags": ["cytotoxic granules", "granule exocytosis", "target-cell killing"],
        "pathway_tags": ["lymphocyte degranulation", "inflammatory signaling"],
        "disease_tags": ["cancer", "parasitic infection", "inflammation"],
        "plain": "NKG7 helps organize how killer immune cells release packages containing cell-killing proteins.",
    },
    "CST7": {
        "official_name": "cystatin F",
        "immune_function": "Immune-enriched cysteine-protease inhibitor that can regulate cathepsins involved in cytotoxic-cell effector pathways.",
        "immune_cells": "NK cells; cytotoxic T cells; myeloid cells and microglia in context-dependent states",
        "biological_role": "Reported to inhibit cathepsin C and other proteases; direction and consequence depend on cell type, processing, and tissue context.",
        "function_tags": ["protease inhibition", "cytotoxic-cell regulation"],
        "pathway_tags": ["cathepsin regulation", "immune effector protease processing"],
        "disease_tags": ["glioblastoma", "viral neuroinflammation"],
        "plain": "CST7 makes cystatin F, a protein that can act like a brake on enzymes used by several immune cells.",
    },
    "SPON2": {
        "official_name": "spondin 2",
        "immune_function": "Extracellular-matrix protein, also called mindin, studied in innate pattern recognition, leukocyte adhesion, and phagocytosis.",
        "immune_cells": "Macrophages and dendritic-cell interactions are best represented in the selected literature; direct NK-cell evidence is limited.",
        "biological_role": "Reported as an extracellular ligand and pattern-recognition molecule in experimental innate-immune models.",
        "function_tags": ["pattern recognition", "cell adhesion", "phagocytosis"],
        "pathway_tags": ["innate pathogen recognition", "integrin signaling"],
        "disease_tags": ["bacterial infection", "innate inflammation"],
        "plain": "SPON2 makes an outside-the-cell protein that can help innate immune cells recognize and interact with microbes; its NK-cell role is uncertain.",
    },
    "GZMA": {
        "official_name": "granzyme A",
        "immune_function": "Cytotoxic-lymphocyte serine protease with reported inflammatory and target-cell death activities that differ from granzyme B.",
        "immune_cells": "NK cells; cytotoxic CD8 T cells",
        "biological_role": "Granule-released protease; selected studies report context-dependent substrate cleavage and inflammatory or cell-death outcomes.",
        "function_tags": ["cytotoxic granules", "serine protease", "target-cell response"],
        "pathway_tags": ["gasdermin-mediated pyroptosis", "lymphocyte cytotoxicity"],
        "disease_tags": ["cancer", "inflammation"],
        "plain": "GZMA makes granzyme A, another enzyme carried by killer immune cells; it does not always act in the same way as granzyme B.",
    },
    "CCL4": {
        "official_name": "C-C motif chemokine ligand 4",
        "immune_function": "Secreted chemokine associated with immune-cell communication and recruitment through chemokine receptors including CCR5.",
        "immune_cells": "Activated NK cells; T cells; additional leukocytes depending on context",
        "biological_role": "Acts as a signaling ligand that can help coordinate leukocyte trafficking rather than directly executing target-cell killing.",
        "function_tags": ["chemokine signaling", "immune-cell recruitment", "cell communication"],
        "pathway_tags": ["CCR5 chemokine axis", "leukocyte trafficking"],
        "disease_tags": ["HIV", "HBV", "inflammation"],
        "plain": "CCL4 is a chemical message that can help immune cells call or guide other immune cells to an area.",
    },
    "CTSW": {
        "official_name": "cathepsin W",
        "immune_function": "Lysosomal cysteine protease enriched in NK cells and cytotoxic T cells; its indispensable substrates and mechanism remain uncertain.",
        "immune_cells": "NK cells; cytotoxic CD8 T cells",
        "biological_role": "Human cell studies report cytotoxic-lymphocyte expression and secretion during target contact, while knockout evidence questions whether it is essential for killing.",
        "function_tags": ["lysosomal protease", "cytotoxic lymphocyte marker"],
        "pathway_tags": ["cysteine-protease regulation", "lymphocyte cytotoxicity"],
        "disease_tags": ["immune pathology (limited evidence)"],
        "plain": "CTSW makes cathepsin W, an enzyme common in killer immune cells, but scientists have not fully settled exactly what it must do.",
    },
}

# Curated after PubMed-first relevance screening. Metadata are never typed by hand.
CURATED_PMIDS = {
    "GZMB": ["38846935", "39179536", "32114394", "38729924"],
    "FGFBP2": ["31801909", "38065972", "38604154"],
    "GNLY": ["14499265", "30658247", "32822574"],
    "PRF1": ["12060139", "14757862", "32542393"],
    "NKG7": ["32839608", "35013002", "34911739"],
    "CST7": ["18256700", "34189679", "38879499"],
    "SPON2": ["14691481", "19153605", "30869196"],
    "GZMA": ["32299851", "32093590", "36792800"],
    "CCL4": ["15354873", "28883824", "38687604"],
    "CTSW": ["19100676", "15087452", "38891048"],
}

ARTICLE_NOTES = {
    ("GZMB", "38846935"): ("Review describes perforin-dependent and perforin-independent granzyme B activity across immune and pathological contexts.", "Mechanism; Association"),
    ("GZMB", "39179536"): ("Review organizes granzyme B with perforin and granulysin in the cytotoxic arsenal used by NK cells.", "Mechanism; Review"),
    ("GZMB", "32114394"): ("Human blood single-cell study reports depletion of a cytotoxic NK-cell subset in tuberculosis and uses cytotoxic-effector genes to define it.", "Association; Biomarker"),
    ("GZMB", "38729924"): ("Human NK-cell cryopreservation experiments report granzyme B-associated apoptosis and test cytokine pretreatment as a protective intervention.", "Mechanism; Experimental evidence"),
    ("FGFBP2", "31801909"): ("Human melanoma single-cell study identifies specialized blood and tumor NK populations using FGFBP2 and CD16-related markers.", "Association; Biomarker"),
    ("FGFBP2", "38065972"): ("Human single-cell and spatial melanoma analysis reports FGFBP2-positive cytotoxic lymphocyte states across clinical tissue groups.", "Association; Biomarker"),
    ("FGFBP2", "38604154"): ("Single-cell analysis of human hepatocellular carcinoma labels an FGFBP2-positive NK subset and reports disease-associated state differences.", "Association; Computational evidence"),
    ("GNLY", "14499265"): ("Review summarizes granulysin structure, processing, antimicrobial activity, and cytotoxic activity.", "Mechanism; Review"),
    ("GNLY", "30658247"): ("Human lymphocyte experiments report that granulysin forms occupy different effector vesicles and can follow different release routes.", "Mechanism; Experimental evidence"),
    ("GNLY", "32822574"): ("Human decidual and peripheral NK-cell experiments report granulysin transfer that kills intracellular bacteria while sparing host cells.", "Mechanism; Experimental evidence"),
    ("PRF1", "12060139"): ("Study of affected families links PRF1 variants to absent perforin expression and impaired cytotoxic function in familial HLH.", "Association; Mechanism; Experimental evidence"),
    ("PRF1", "14757862"): ("Multicenter family study characterizes diverse PRF1 mutations associated with reduced NK-cell activity in familial HLH.", "Association; Experimental evidence"),
    ("PRF1", "32542393"): ("Large clinical referral cohort reports PRF1 among the most frequent genes with pathogenic findings in suspected genetic HLH.", "Association; Biomarker"),
    ("NKG7", "32839608"): ("Human and experimental models report that NKG7 regulates cytotoxic-granule exocytosis, target killing, and inflammatory responses.", "Mechanism; Experimental evidence"),
    ("NKG7", "35013002"): ("Experimental cancer models report that NKG7 is required for optimal antitumor T-cell activity.", "Mechanism; Experimental evidence"),
    ("NKG7", "34911739"): ("Human-tumor analyses and laboratory experiments examine NKG7 as a target for improving T-cell cytotoxic function.", "Association; Experimental evidence"),
    ("CST7", "18256700"): ("Biochemical and human immune-cell experiments identify cystatin F as a cathepsin C-directed inhibitor enriched in cytotoxic lymphocytes.", "Mechanism; Experimental evidence"),
    ("CST7", "34189679"): ("Glioblastoma study reports cystatin F expression and transfer in tumor and immune cells alongside reduced NK-cell cytotoxic susceptibility.", "Association; Mechanism; Experimental evidence"),
    ("CST7", "38879499"): ("Mouse coronavirus study reports altered neuroinflammation and immune-cell transcription after Cst7 deletion.", "Mechanism; Experimental evidence"),
    ("SPON2", "14691481"): ("Mouse experiments report mindin as an extracellular pattern-recognition molecule needed for normal responses to microbial challenge.", "Mechanism; Experimental evidence"),
    ("SPON2", "19153605"): ("Structural study examines the mindin domain involved in integrin binding and pattern recognition.", "Mechanism; Experimental evidence"),
    ("SPON2", "30869196"): ("Experimental macrophage study reports mindin binding to Mac-1 and promotion of phagocytosis-related signaling.", "Mechanism; Experimental evidence"),
    ("GZMA", "32299851"): ("Human-cell and mouse experiments report granzyme A cleavage of GSDMB and target-cell pyroptosis under specified conditions.", "Mechanism; Experimental evidence"),
    ("GZMA", "32093590"): ("Review compares granzyme effects involving mitochondria and emphasizes distinct mechanisms among granzyme family members.", "Mechanism; Review"),
    ("GZMA", "36792800"): ("Ex vivo human blood NK-cell study reports reduced granzyme A and other effector outputs after butyrate exposure.", "Mechanism; Experimental evidence"),
    ("CCL4", "15354873"): ("Review organizes chemokines and receptors involved in NK-cell movement between blood and tissues.", "Mechanism; Review"),
    ("CCL4", "28883824"): ("Human study reports that NK-cell education influences antibody-dependent activation, cytotoxicity, and chemokine responses during HIV-related experiments.", "Association; Experimental evidence"),
    ("CCL4", "38687604"): ("Human HIV/HBV coinfection study reports altered NK-cell functional responses, including chemokine-associated measurements.", "Association; Biomarker"),
    ("CTSW", "19100676"): ("Human cytotoxic-lymphocyte study reports cathepsin W secretion during target-cell contact but finds it nonessential in tested CTL killing assays.", "Mechanism; Experimental evidence"),
    ("CTSW", "15087452"): ("Mouse knockout study characterizes cathepsin W expression and tests its contribution to cell-mediated cytotoxicity.", "Mechanism; Experimental evidence"),
    ("CTSW", "38891048"): ("Review summarizes cysteine proteases F and W, including immune expression, proposed roles, pathology, and unresolved therapeutic questions.", "Association; Review"),
}

missing_profiles = set(pilot_genes["gene"]) - set(GENE_PROFILES)
missing_searches = set(pilot_genes["gene"]) - set(CURATED_PMIDS)
if missing_profiles or missing_searches:
    raise ValueError(f"Unconfigured pilot genes. Profiles={missing_profiles}; searches={missing_searches}")
print(f"Configured literature evidence for {len(pilot_genes)} pilot genes and {sum(map(len, CURATED_PMIDS.values()))} gene–publication links.")

Configured literature evidence for 10 pilot genes and 31 gene–publication links.


## 3. Retrieve and verify PubMed records

The NCBI response supplies PMID, DOI, journal, year, title, abstract, MeSH terms, and publication types. Missing DOI values remain blank and are explicitly displayed as “not listed”; they are never guessed.

In [4]:
def node_text(node):
    return "" if node is None else "".join(node.itertext()).strip()

def parse_year(article):
    for path in [
        ".//JournalIssue/PubDate/Year", ".//ArticleDate/Year", ".//PubDate/MedlineDate"
    ]:
        text = node_text(article.find(path))
        match = re.search(r"(?:19|20)\d{2}", text)
        if match:
            return int(match.group())
    return pd.NA

def fetch_pubmed_records(pmids, attempts=3):
    endpoint = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "xml", "tool": "pbmc3k_phase7"}
    headers = {"User-Agent": "PBMC3k educational literature integration notebook"}
    for attempt in range(attempts):
        response = requests.get(endpoint, params=params, headers=headers, timeout=90)
        if response.status_code == 200:
            break
        if response.status_code == 429 and attempt < attempts - 1:
            time.sleep(2 ** (attempt + 1))
            continue
        response.raise_for_status()
    root = ET.fromstring(response.content)
    records = []
    for citation in root.findall(".//PubmedArticle"):
        medline = citation.find("MedlineCitation")
        article = medline.find("Article")
        pmid = node_text(medline.find("PMID"))
        doi = ""
        for article_id in citation.findall(".//PubmedData/ArticleIdList/ArticleId"):
            if article_id.attrib.get("IdType") == "doi":
                doi = node_text(article_id)
                break
        abstract_parts = []
        for part in article.findall(".//Abstract/AbstractText"):
            label = part.attrib.get("Label", "")
            text = node_text(part)
            abstract_parts.append(f"{label}: {text}" if label else text)
        publication_types = [node_text(x) for x in article.findall(".//PublicationTypeList/PublicationType")]
        mesh_terms = [node_text(x) for x in medline.findall(".//MeshHeading/DescriptorName")]
        records.append({
            "PMID": pmid,
            "title": node_text(article.find("ArticleTitle")),
            "journal": node_text(article.find("Journal/Title")),
            "year": parse_year(article),
            "DOI": doi,
            "abstract": " ".join(abstract_parts),
            "publication_types": "; ".join(publication_types),
            "mesh_terms": "; ".join(mesh_terms),
        })
    return pd.DataFrame(records)

def classify_study(row):
    text = " ".join([row["title"], row["abstract"], row["publication_types"], row["mesh_terms"]]).lower()
    is_review = "review" in row["publication_types"].lower()
    is_human = "humans" in row["mesh_terms"].lower()
    is_animal = any(term in row["mesh_terms"].lower() for term in ["mice", "animals"])
    is_single_cell = any(term in text for term in ["single-cell", "single cell", "scrna-seq", "spatial transcript"])
    is_computational = any(term in text for term in ["bioinformatic", "database", "machine learning", "network pharmacology"])
    if is_review:
        return "Review"
    if is_human and is_single_cell:
        return "Human single-cell/transcriptomic study"
    if is_human and is_animal:
        return "Human and laboratory/animal study"
    if is_human:
        return "Human study"
    if is_animal:
        return "Laboratory/animal study"
    if is_computational:
        return "Computational study"
    return "Laboratory or other primary study"

unique_pmids = sorted({pmid for values in CURATED_PMIDS.values() for pmid in values})
pubmed = fetch_pubmed_records(unique_pmids)
missing_pmids = set(unique_pmids) - set(pubmed["PMID"])
if missing_pmids:
    raise RuntimeError(f"PubMed did not return verified records for PMIDs: {sorted(missing_pmids)}")
if pubmed[["PMID", "title", "journal", "year"]].isna().any().any() or pubmed["title"].eq("").any():
    raise RuntimeError("At least one verified PubMed record is missing core metadata.")
pubmed["study_type"] = pubmed.apply(classify_study, axis=1)

# Publication-level review prevents broad MeSH tags from overstating human evidence.
STUDY_TYPE_OVERRIDES = {
    "38846935": "Review", "39179536": "Review", "32114394": "Human single-cell/transcriptomic study", "38729924": "Human ex vivo/laboratory study",
    "31801909": "Human single-cell/transcriptomic study", "38065972": "Human single-cell/transcriptomic study", "38604154": "Human single-cell/transcriptomic study",
    "14499265": "Review", "30658247": "Human laboratory study", "32822574": "Human ex vivo/laboratory study",
    "12060139": "Human family study", "14757862": "Human multicenter family study", "32542393": "Human clinical cohort",
    "32839608": "Human and laboratory/animal study", "35013002": "Laboratory/animal study", "34911739": "Human and laboratory study",
    "18256700": "Human laboratory study", "34189679": "Human and laboratory study", "38879499": "Laboratory/animal study",
    "14691481": "Laboratory/animal study", "19153605": "Structural laboratory study", "30869196": "Laboratory/animal study",
    "32299851": "Human and laboratory/animal study", "32093590": "Review", "36792800": "Human ex vivo study",
    "15354873": "Review", "28883824": "Human study", "38687604": "Human study",
    "19100676": "Human laboratory study", "15087452": "Laboratory/animal study", "38891048": "Review",
}
pubmed["study_type"] = pubmed.apply(
    lambda row: STUDY_TYPE_OVERRIDES.get(row["PMID"], row["study_type"]), axis=1
)

links = pd.DataFrame(
    [(gene, pmid, *ARTICLE_NOTES[(gene, pmid)]) for gene, pmids in CURATED_PMIDS.items() for pmid in pmids],
    columns=["gene", "PMID", "summary", "evidence_categories"],
)
references = links.merge(pubmed, on="PMID", how="left", validate="many_to_one")
display(references[["gene", "PMID", "title", "journal", "year", "DOI", "study_type"]])

,gene,PMID,title,journal,year,DOI,study_type
0,GZMB,38846935,Reassessing granzyme B: unveiling perforin-independent versatility in immune responses and therapeutic potentials.,Frontiers in immunology,2024,10.3389/fimmu.2024.1392535,Review
1,GZMB,39179536,Natural Killer cells at the frontline in the fight against cancer.,Cell death & disease,2024,10.1038/s41419-024-06976-0,Review
2,GZMB,32114394,Single-cell transcriptomics of blood reveals a natural killer cell subset depletion in tuberculosis.,EBioMedicine,2020,10.1016/j.ebiom.2020.102686,Human single-cell/transcriptomic study
3,GZMB,38729924,Pretreatment with IL-15 and IL-18 rescues natural killer cells from granzyme B-mediated apoptosis after cryopreserva...,Nature communications,2024,10.1038/s41467-024-47574-0,Human ex vivo/laboratory study
4,FGFBP2,31801909,Discovery of specialized NK cell populations infiltrating human melanoma metastases.,JCI insight,2019,10.1172/jci.insight.133103,Human single-cell/transcriptomic study
5,FGFBP2,38065972,Delineating the early dissemination mechanisms of acral melanoma by integrating single-cell and spatial transcriptom...,Nature communications,2023,10.1038/s41467-023-43980-y,Human single-cell/transcriptomic study
6,FGFBP2,38604154,Single-cell data revealed exhaustion of characteristic NK cell subpopulations and T cell subpopulations in hepatocel...,Aging,2024,10.18632/aging.205723,Human single-cell/transcriptomic study
7,GNLY,14499265,Granulysin.,Current opinion in immunology,2003,10.1016/s0952-7915(03)00097-9,Review
8,GNLY,30658247,Granulysin species segregate to different lysosome-related effector vesicles (LREV) and get mobilized by either clas...,Molecular immunology,2019,10.1016/j.molimm.2018.12.031,Human laboratory study
9,GNLY,32822574,Decidual NK Cells Transfer Granulysin to Selectively Kill Bacteria in Trophoblasts.,Cell,2020,10.1016/j.cell.2020.07.019,Human ex vivo/laboratory study


## 4. Assign transparent evidence grades

Grades describe the strength of the **selected literature collection for this gene**, not the importance of the gene and not the certainty of every claim.

- **A — Repeated human evidence:** at least three primary human studies.
- **B — Multiple human studies:** at least two primary human studies.
- **C — Laboratory or animal evidence:** experimental evidence exists, but human repetition does not meet A/B.
- **D — Computational evidence:** the collection is computational without stronger primary experimental evidence.
- **E — Limited evidence:** none of the above thresholds is met.

A review is useful context but is not counted as a primary human study.

In [5]:
def assign_grade(group):
    types = group["study_type"]
    human_primary = int(types.str.startswith("Human").sum())
    reviews = int(types.eq("Review").sum())
    experimental = int(types.str.contains("Laboratory|animal|primary", case=False, regex=True).sum())
    computational = int(types.str.contains("Computational", case=False).sum())
    if human_primary >= 3:
        grade = "A"
        reason = f"Repeated human evidence: {human_primary} primary human studies; {reviews} review(s) provide context but do not determine the grade."
    elif human_primary >= 2:
        grade = "B"
        reason = f"Multiple human studies: {human_primary} primary human studies; {reviews} review(s) provide context but do not determine the grade."
    elif experimental >= 1:
        grade = "C"
        reason = f"Laboratory or animal evidence predominates ({experimental} selected experimental study/studies); repeated human evidence is insufficient."
    elif computational >= 1:
        grade = "D"
        reason = "Selected evidence is computational and lacks stronger primary experimental support in this collection."
    else:
        grade = "E"
        reason = "Selected evidence is limited and does not meet the thresholds for grades A–D."
    return pd.Series({"evidence_grade": grade, "grade_explanation": reason})

grades = references.groupby("gene", sort=False).apply(assign_grade, include_groups=False).reset_index()
references = references.merge(grades, on="gene", how="left", validate="many_to_one")
display(grades)

,gene,evidence_grade,grade_explanation
0,GZMB,B,Multiple human studies: 2 primary human studies; 2 review(s) provide context but do not determine the grade.
1,FGFBP2,A,Repeated human evidence: 3 primary human studies; 0 review(s) provide context but do not determine the grade.
2,GNLY,B,Multiple human studies: 2 primary human studies; 1 review(s) provide context but do not determine the grade.
3,PRF1,A,Repeated human evidence: 3 primary human studies; 0 review(s) provide context but do not determine the grade.
4,NKG7,B,Multiple human studies: 2 primary human studies; 0 review(s) provide context but do not determine the grade.
5,CST7,B,Multiple human studies: 2 primary human studies; 0 review(s) provide context but do not determine the grade.
6,SPON2,C,Laboratory or animal evidence predominates (3 selected experimental study/studies); repeated human evidence is insuf...
7,GZMA,B,Multiple human studies: 2 primary human studies; 1 review(s) provide context but do not determine the grade.
8,CCL4,B,Multiple human studies: 2 primary human studies; 1 review(s) provide context but do not determine the grade.
9,CTSW,C,Laboratory or animal evidence predominates (2 selected experimental study/studies); repeated human evidence is insuf...


## 5. Build machine-readable and human-readable outputs

Each gene file begins with the Phase 6 observation, then presents publication-specific evidence. Disease-related material is separated into association, mechanism, biomarker, and experimental-evidence categories. These headings organize reported contexts; they do not imply that every gene has evidence in every category.

In [6]:
def md_table(frame):
    shown = frame.fillna("").astype(str)
    headers = list(shown.columns)
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in shown.itertuples(index=False, name=None):
        clean = [str(value).replace("|", "\\|").replace("\n", " ") for value in row]
        lines.append("| " + " | ".join(clean) + " |")
    return "\n".join(lines)

def disease_section(gene_refs):
    category_order = ["Association", "Mechanism", "Biomarker", "Experimental evidence"]
    lines = []
    for category in category_order:
        subset = gene_refs[gene_refs["evidence_categories"].str.contains(category, case=False, regex=False)]
        lines.append(f"### {category}\n")
        if subset.empty:
            lines.append("No publication in the selected evidence set was assigned to this category.\n")
        else:
            for row in subset.itertuples():
                lines.append(f"- PMID {row.PMID}: {row.summary}\n")
    return "\n".join(lines)

summary_rows = []
gene_files = []
for marker in pilot_genes.itertuples(index=False):
    gene = marker.gene
    profile = GENE_PROFILES[gene]
    gene_refs = references[references["gene"] == gene].copy()
    grade = gene_refs.iloc[0]["evidence_grade"]
    grade_explanation = gene_refs.iloc[0]["grade_explanation"]

    literature_blocks = []
    for ref in gene_refs.itertuples(index=False):
        doi_text = ref.DOI if ref.DOI else "not listed in the PubMed record"
        literature_blocks.append(
            f"### {ref.title}\n\n"
            f"- **Journal:** {ref.journal}\n"
            f"- **Year:** {ref.year}\n"
            f"- **PMID:** [{ref.PMID}](https://pubmed.ncbi.nlm.nih.gov/{ref.PMID}/)\n"
            f"- **DOI:** {doi_text}\n"
            f"- **Study type:** {ref.study_type}\n"
            f"- **Organizing summary:** {ref.summary}\n"
        )

    gene_report = f"""# {gene} — Literature Evidence File

> Scope: evidence collection only. Dataset observations, publication findings, and future interpretation are kept separate.

## Dataset Evidence

- **Cluster:** {marker.cluster} ({marker.cell_type})
- **Representative-gene rank:** {int(marker.representative_rank)}
- **Marker score:** {marker.marker_score:.3f}
- **Average log2 fold change:** {marker.avg_log2FC:.3f}
- **Percent expressed in cluster:** {marker.pct_in:.1%}
- **Percent expressed outside cluster:** {marker.pct_out:.1%}
- **Specificity difference:** {marker.specificity_delta:.3f}

These are observations from this PBMC3k analysis, not literature claims.

## Biological Function

- **Official gene name:** {profile['official_name']}
- **Immune function:** {profile['immune_function']}
- **Common immune-cell contexts:** {profile['immune_cells']}
- **Reported biological role:** {profile['biological_role']}

## Literature Evidence

{chr(10).join(literature_blocks)}

## Disease Associations

{disease_section(gene_refs)}

**Safety statement:** These are contexts reported by publications. Expression of {gene} in this dataset does not diagnose, predict, or demonstrate any disease.

## Evidence Grade

**{grade}** — {grade_explanation}

The grade applies only to this selected literature collection and may change as evidence is added.

## Limitations

- Association is not diagnosis and does not establish causality.
- Gene expression is context-dependent and can change with cell state or stimulation.
- Evidence from tumors, infected tissue, animal models, or cell lines may not transfer to healthy peripheral blood.
- Cell-type and tissue specificity limit generalization.
- A marker can identify a cell state without being the mechanism that creates that state.
- This is a focused evidence set, not a formal systematic review or meta-analysis.

## Plain Language Notes

{profile['plain']}

Finding this gene in the PBMC3k NK cluster helps describe the cells. By itself, it cannot reveal a person's health, identity, or future.

## Deferred to Notebook 08

No combined biological conclusion is made here. Future synthesis must preserve uncertainty and distinguish this dataset from the cited study populations.
"""
    gene_path = GENE_DIR / f"{gene}.md"
    gene_path.write_text(gene_report, encoding="utf-8")
    gene_files.append(gene_path)
    summary_rows.append({
        "gene": gene,
        "cluster": marker.cluster,
        "cell_type": marker.cell_type,
        "representative_rank": int(marker.representative_rank),
        "marker_score": marker.marker_score,
        "avg_log2FC": marker.avg_log2FC,
        "pct_in": marker.pct_in,
        "pct_out": marker.pct_out,
        "official_gene_name": profile["official_name"],
        "immune_function": profile["immune_function"],
        "immune_cell_contexts": profile["immune_cells"],
        "biological_role": profile["biological_role"],
        "function_tags": "; ".join(profile["function_tags"]),
        "pathway_tags": "; ".join(profile["pathway_tags"]),
        "disease_context_tags": "; ".join(profile["disease_tags"]),
        "publication_count": len(gene_refs),
        "evidence_grade": grade,
        "grade_explanation": grade_explanation,
        "plain_language_note": profile["plain"],
        "interpretation_status": "Deferred to Notebook 08",
    })

literature_summary = pd.DataFrame(summary_rows).sort_values("representative_rank")
literature_summary_path = PHASE7 / "literature_summary.csv"
literature_summary.to_csv(literature_summary_path, index=False)

reference_columns = [
    "gene", "title", "journal", "year", "PMID", "DOI", "evidence_grade", "study_type",
    "summary", "evidence_categories", "publication_types", "mesh_terms",
]
references_path = PHASE7 / "references.csv"
references[reference_columns].sort_values(["gene", "year", "PMID"]).to_csv(references_path, index=False)

print(f"Wrote {len(gene_files)} gene reports, {len(literature_summary)} summary rows, and {len(references)} gene–publication reference rows.")

Wrote 10 gene reports, 10 summary rows, and 31 gene–publication reference rows.


## 6. Cluster reference report

The report below is an index of recurring vocabulary and study contexts. Counts indicate how many gene profiles were assigned each tag; they are not pathway-enrichment statistics and are not a combined interpretation.

In [7]:
def tag_counts(field):
    return Counter(tag for gene in pilot_genes["gene"] for tag in GENE_PROFILES[gene][field])

def count_lines(counts):
    return "\n".join(f"- {term}: tagged for {count} gene(s)" for term, count in counts.most_common())

gene_grade_table = literature_summary[["gene", "evidence_grade", "publication_count", "grade_explanation"]].copy()
gene_grade_table.columns = ["Gene", "Grade", "References", "Reason"]
unanswered = [
    "Which selected markers are stable across independent healthy PBMC cohorts?",
    "Which markers change with NK-cell activation, maturation, or recent target-cell contact?",
    "Does FGFBP2 have a direct immune mechanism, or is it primarily a correlated cytotoxic-state marker?",
    "What direct role, if any, does SPON2 have in circulating human NK cells?",
    "Which substrates make CTSW functionally important in human NK cells, and under what conditions?",
    "How well do tumor, infection, animal, and cell-line findings transfer to healthy peripheral blood?",
    "Which claims have been independently replicated with both protein-level and functional measurements?",
]

cluster_report = f"""# Literature Reference Report — Cluster {pilot_cluster}: {pilot_cell_type}

> This document organizes evidence for later analysis. It does not interpret the genes as a combined program and does not make a biological or clinical conclusion.

## Selected Genes

{', '.join(pilot_genes['gene'])}

## Dataset Boundary

The genes were selected from the Phase 6 marker ranking for cluster {pilot_cluster}. Marker statistics are dataset observations. All publication claims come from the verified references table. Disease contexts in those papers do not describe or diagnose the PBMC3k donor.

## Recurring Biological Functions

{count_lines(tag_counts('function_tags'))}

## Recurring Immune Pathways

{count_lines(tag_counts('pathway_tags'))}

## Recurring Disease Themes in the Selected Publications

{count_lines(tag_counts('disease_tags'))}

These are indexing tags, not evidence of disease in this dataset and not formal enrichment results.

## Confidence of Literature

{md_table(gene_grade_table)}

Grades describe the selected evidence set using the explicit A–E rules in Notebook 07. A high grade does not make every reported mechanism universal or causal.

## Unanswered Biological Questions

{chr(10).join('- ' + question for question in unanswered)}

## Future Interpretation

Cross-gene reasoning, pathway synthesis, and biological conclusions are intentionally deferred to Notebook 08.
"""
cluster_report_path = PHASE7 / "cluster_reference_report.md"
cluster_report_path.write_text(cluster_report, encoding="utf-8")
display(Markdown(cluster_report))

# Literature Reference Report — Cluster 5: NK cells

> This document organizes evidence for later analysis. It does not interpret the genes as a combined program and does not make a biological or clinical conclusion.

## Selected Genes

GZMB, FGFBP2, GNLY, PRF1, NKG7, CST7, SPON2, GZMA, CCL4, CTSW

## Dataset Boundary

The genes were selected from the Phase 6 marker ranking for cluster 5. Marker statistics are dataset observations. All publication claims come from the verified references table. Disease contexts in those papers do not describe or diagnose the PBMC3k donor.

## Recurring Biological Functions

- cytotoxic granules: tagged for 5 gene(s)
- target-cell killing: tagged for 4 gene(s)
- serine protease: tagged for 2 gene(s)
- cytotoxic cell-state marker: tagged for 1 gene(s)
- NK-cell heterogeneity: tagged for 1 gene(s)
- antimicrobial defense: tagged for 1 gene(s)
- pore formation: tagged for 1 gene(s)
- granule exocytosis: tagged for 1 gene(s)
- protease inhibition: tagged for 1 gene(s)
- cytotoxic-cell regulation: tagged for 1 gene(s)
- pattern recognition: tagged for 1 gene(s)
- cell adhesion: tagged for 1 gene(s)
- phagocytosis: tagged for 1 gene(s)
- target-cell response: tagged for 1 gene(s)
- chemokine signaling: tagged for 1 gene(s)
- immune-cell recruitment: tagged for 1 gene(s)
- cell communication: tagged for 1 gene(s)
- lysosomal protease: tagged for 1 gene(s)
- cytotoxic lymphocyte marker: tagged for 1 gene(s)

## Recurring Immune Pathways

- lymphocyte cytotoxicity: tagged for 3 gene(s)
- perforin–granzyme pathway: tagged for 2 gene(s)
- programmed cell death: tagged for 1 gene(s)
- cytotoxic lymphocyte program: tagged for 1 gene(s)
- cytotoxic lymphocyte degranulation: tagged for 1 gene(s)
- host defense: tagged for 1 gene(s)
- lymphocyte degranulation: tagged for 1 gene(s)
- inflammatory signaling: tagged for 1 gene(s)
- cathepsin regulation: tagged for 1 gene(s)
- immune effector protease processing: tagged for 1 gene(s)
- innate pathogen recognition: tagged for 1 gene(s)
- integrin signaling: tagged for 1 gene(s)
- gasdermin-mediated pyroptosis: tagged for 1 gene(s)
- CCR5 chemokine axis: tagged for 1 gene(s)
- leukocyte trafficking: tagged for 1 gene(s)
- cysteine-protease regulation: tagged for 1 gene(s)

## Recurring Disease Themes in the Selected Publications

- inflammation: tagged for 4 gene(s)
- cancer: tagged for 3 gene(s)
- bacterial infection: tagged for 2 gene(s)
- infection: tagged for 1 gene(s)
- melanoma: tagged for 1 gene(s)
- hepatocellular carcinoma: tagged for 1 gene(s)
- pregnancy infection defense: tagged for 1 gene(s)
- familial hemophagocytic lymphohistiocytosis: tagged for 1 gene(s)
- parasitic infection: tagged for 1 gene(s)
- glioblastoma: tagged for 1 gene(s)
- viral neuroinflammation: tagged for 1 gene(s)
- innate inflammation: tagged for 1 gene(s)
- HIV: tagged for 1 gene(s)
- HBV: tagged for 1 gene(s)
- immune pathology (limited evidence): tagged for 1 gene(s)

These are indexing tags, not evidence of disease in this dataset and not formal enrichment results.

## Confidence of Literature

| Gene | Grade | References | Reason |
| --- | --- | --- | --- |
| GZMB | B | 4 | Multiple human studies: 2 primary human studies; 2 review(s) provide context but do not determine the grade. |
| FGFBP2 | A | 3 | Repeated human evidence: 3 primary human studies; 0 review(s) provide context but do not determine the grade. |
| GNLY | B | 3 | Multiple human studies: 2 primary human studies; 1 review(s) provide context but do not determine the grade. |
| PRF1 | A | 3 | Repeated human evidence: 3 primary human studies; 0 review(s) provide context but do not determine the grade. |
| NKG7 | B | 3 | Multiple human studies: 2 primary human studies; 0 review(s) provide context but do not determine the grade. |
| CST7 | B | 3 | Multiple human studies: 2 primary human studies; 0 review(s) provide context but do not determine the grade. |
| SPON2 | C | 3 | Laboratory or animal evidence predominates (3 selected experimental study/studies); repeated human evidence is insufficient. |
| GZMA | B | 3 | Multiple human studies: 2 primary human studies; 1 review(s) provide context but do not determine the grade. |
| CCL4 | B | 3 | Multiple human studies: 2 primary human studies; 1 review(s) provide context but do not determine the grade. |
| CTSW | C | 3 | Laboratory or animal evidence predominates (2 selected experimental study/studies); repeated human evidence is insufficient. |

Grades describe the selected evidence set using the explicit A–E rules in Notebook 07. A high grade does not make every reported mechanism universal or causal.

## Unanswered Biological Questions

- Which selected markers are stable across independent healthy PBMC cohorts?
- Which markers change with NK-cell activation, maturation, or recent target-cell contact?
- Does FGFBP2 have a direct immune mechanism, or is it primarily a correlated cytotoxic-state marker?
- What direct role, if any, does SPON2 have in circulating human NK cells?
- Which substrates make CTSW functionally important in human NK cells, and under what conditions?
- How well do tumor, infection, animal, and cell-line findings transfer to healthy peripheral blood?
- Which claims have been independently replicated with both protein-level and functional measurements?

## Future Interpretation

Cross-gene reasoning, pathway synthesis, and biological conclusions are intentionally deferred to Notebook 08.


## 7. Validation and output manifest

In [8]:
required_reference_columns = {"gene", "title", "journal", "year", "PMID", "DOI", "evidence_grade", "study_type"}
expected_outputs = [literature_summary_path, references_path, cluster_report_path, *gene_files]
manifest = pd.DataFrame({
    "output": [str(path.relative_to(PROJECT)) for path in expected_outputs],
    "exists": [path.exists() for path in expected_outputs],
    "size_bytes": [path.stat().st_size if path.exists() else 0 for path in expected_outputs],
})

assert manifest["exists"].all(), "At least one required Phase 7 output is missing."
assert required_reference_columns.issubset(references.columns), "references.csv schema is incomplete."
assert references["PMID"].astype(str).str.fullmatch(r"\d+").all(), "A PMID is malformed."
assert references["title"].str.len().gt(0).all(), "A verified title is missing."
assert references["DOI"].fillna("").map(lambda x: x == "" or "/" in x or x.startswith("10.")).all(), "A DOI has an unexpected format."
assert set(pilot_genes["gene"]) == set(literature_summary["gene"]), "Not every pilot gene has a summary."
assert set(pilot_genes["gene"]) == set(references["gene"]), "Not every pilot gene has literature evidence."
assert references.groupby("gene")["PMID"].nunique().ge(1).all(), "A gene has no verified PubMed reference."
assert literature_summary["interpretation_status"].eq("Deferred to Notebook 08").all()

display(manifest)
print("All Phase 7 citation, schema, coverage, and output checks passed.")

,output,exists,size_bytes
0,results/phase7/literature_summary.csv,True,8396
1,results/phase7/references.csv,True,19527
2,results/phase7/cluster_reference_report.md,True,5471
3,results/phase7/genes/GZMB.md,True,5367
4,results/phase7/genes/FGFBP2.md,True,4891
5,results/phase7/genes/GNLY.md,True,4619
6,results/phase7/genes/PRF1.md,True,4862
7,results/phase7/genes/NKG7.md,True,4625
8,results/phase7/genes/CST7.md,True,4979
9,results/phase7/genes/SPON2.md,True,4883


All Phase 7 citation, schema, coverage, and output checks passed.


## 8. Extend verified evidence coverage to every Phase 6 cluster

The validated Cluster 5 workflow above remains the template and is executed first. The extension below:

- detects every cluster and representative gene from Phase 6;
- preserves the validated Cluster 5 summary rows, reference rows, and gene files;
- reuses verified gene-level evidence only when the same gene occurs in another cluster;
- performs gene-specific PubMed searches for the remaining unique genes;
- verifies all selected PMIDs through PubMed EFetch;
- records an explicit grade-E insufficient-evidence outcome rather than inventing a reference;
- creates coverage, reuse, and validation reports for clusters 0–8.

Publication-specific evidence remains separate from cluster-specific marker observations. Biological synthesis is still deferred to Notebook 08.

In [9]:
import runpy

extension_script = PROJECT / "scripts" / "extend_phase7_all_clusters.py"
if not extension_script.exists():
    raise FileNotFoundError(f"Missing Phase 7 extension script: {extension_script}")

runpy.run_path(str(extension_script), run_name="__main__")

coverage = pd.read_csv(PHASE7 / "phase7_coverage_summary.csv", dtype={"cluster_id": str})
reuse = pd.read_csv(PHASE7 / "evidence_reuse_report.csv")
validation_report = pd.read_json(
    PHASE7 / "phase7_validation_report.json", typ="series"
)

display(coverage)
display(reuse[reuse["evidence_reused"].astype(str).str.lower().eq("true")])
display(validation_report.to_frame("value"))

assert coverage["phase7_status"].eq("COMPLETE").all()
assert validation_report["status"] == "PASS"
print("All-cluster Phase 7 extension completed and validated.")

[01/68] AIF1: 15 PubMed candidate(s)
[02/68] AQP3: 15 PubMed candidate(s)
[03/68] C6orf48: 3 PubMed candidate(s)
[04/68] CCL5: 15 PubMed candidate(s)
[05/68] CCR7: 15 PubMed candidate(s)
[06/68] CD14: 15 PubMed candidate(s)
[07/68] CD2: 15 PubMed candidate(s)
[08/68] CD3D: 15 PubMed candidate(s)
[09/68] CD3E: 15 PubMed candidate(s)
[10/68] CD7: 15 PubMed candidate(s)
[11/68] CD79A: 15 PubMed candidate(s)
[12/68] CD79B: 15 PubMed candidate(s)
[13/68] CD8A: 15 PubMed candidate(s)
[14/68] CD9: 15 PubMed candidate(s)
[15/68] CDKN1C: 15 PubMed candidate(s)
[16/68] CFD: 15 PubMed candidate(s)
[17/68] CST3: 15 PubMed candidate(s)
[18/68] FCER1G: 15 PubMed candidate(s)
[19/68] FCER2: 15 PubMed candidate(s)
[20/68] FCGR3A: 15 PubMed candidate(s)
[21/68] FCN1: 15 PubMed candidate(s)
[22/68] GNG11: 15 PubMed candidate(s)
[23/68] GP9: 15 PubMed candidate(s)
[24/68] GZMH: 15 PubMed candidate(s)
[25/68] GZMK: 15 PubMed candidate(s)
[26/68] HIST1H2AC: 2 PubMed candidate(s)
[27/68] HLA-DQA1: 15 PubMed

Excluding ESearch hits that EFetch did not return as PubMed articles: ['20301499', '20301568', '20641480', '28723059']
AIF1: selected 3 verified reference(s).
AQP3: selected 3 verified reference(s).
C6orf48: selected 3 verified reference(s).
CCL5: selected 3 verified reference(s).
CCR7: selected 3 verified reference(s).
CD14: selected 3 verified reference(s).
CD2: selected 3 verified reference(s).
CD3D: selected 3 verified reference(s).
CD3E: selected 3 verified reference(s).
CD7: selected 3 verified reference(s).
CD79A: selected 3 verified reference(s).
CD79B: selected 3 verified reference(s).
CD8A: selected 3 verified reference(s).
CD9: selected 3 verified reference(s).
CDKN1C: selected 3 verified reference(s).
CFD: selected 3 verified reference(s).
CST3: selected 3 verified reference(s).
FCER1G: selected 3 verified reference(s).
FCER2: selected 3 verified reference(s).
FCGR3A: selected 3 verified reference(s).
FCN1: selected 3 verified reference(s).
GNG11: selected 3 verified refere


Phase 7 completion summary
- Clusters completed: 0, 1, 2, 3, 4, 5, 6, 7, 8
- Total unique genes reviewed: 78
- Total cluster-gene entries covered: 90
- Total verified reference rows: 231
- Total unique verified PMIDs: 224
- Genes with insufficient evidence: RP11-290F20.3
- Validation: PASS
- Remaining gaps: No structural or citation-validation gaps; grade-E genes remain explicitly limited.


,cluster_id,proposed_cell_type,representative_gene_count,genes_with_evidence,genes_with_verified_references,genes_with_insufficient_evidence,verified_reference_count,phase7_status
0,0,Cytotoxic CD8 T cells,10,10,10,0,30,COMPLETE
1,1,B cells,10,10,10,0,29,COMPLETE
2,2,IL7R+ memory/helper T cells,10,10,10,0,28,COMPLETE
3,3,Classical monocytes,10,10,10,0,30,COMPLETE
4,4,CD16+ non-classical monocytes,10,9,9,1,27,COMPLETE
5,5,NK cells,10,10,10,0,31,COMPLETE
6,6,Activated/transitional T cells,10,10,10,0,28,COMPLETE
7,7,Naive/resting T cells,10,10,10,0,29,COMPLETE
8,8,Platelets,10,10,10,0,29,COMPLETE


,gene,clusters_using_gene,evidence_reused,source_evidence_file,notes
0,AIF1,3; 4,True,results/phase7/genes/AIF1.md,The same verified gene-level evidence is reused; each cluster retains its own marker statistics and later interpreta...
4,CCL5,0; 6,True,results/phase7/genes/CCL5.md,The same verified gene-level evidence is reused; each cluster retains its own marker statistics and later interpreta...
8,CD3D,2; 6; 7,True,results/phase7/genes/CD3D.md,The same verified gene-level evidence is reused; each cluster retains its own marker statistics and later interpreta...
9,CD3E,2; 7,True,results/phase7/genes/CD3E.md,The same verified gene-level evidence is reused; each cluster retains its own marker statistics and later interpreta...
18,CST7,0; 5,True,results/phase7/genes/CST7.md,The same verified gene-level evidence is reused; each cluster retains its own marker statistics and later interpreta...
19,CTSW,0; 5,True,results/phase7/genes/CTSW.md,The same verified gene-level evidence is reused; each cluster retains its own marker statistics and later interpreta...
28,GZMA,0; 5,True,results/phase7/genes/GZMA.md,The same verified gene-level evidence is reused; each cluster retains its own marker statistics and later interpreta...
37,IL32,2; 6,True,results/phase7/genes/IL32.md,The same verified gene-level evidence is reused; each cluster retains its own marker statistics and later interpreta...
41,LDHB,2; 7,True,results/phase7/genes/LDHB.md,The same verified gene-level evidence is reused; each cluster retains its own marker statistics and later interpreta...
45,LST1,3; 4,True,results/phase7/genes/LST1.md,The same verified gene-level evidence is reused; each cluster retains its own marker statistics and later interpreta...


,value
status,PASS
errors,[]
valid_clusters,"[0, 1, 2, 3, 4, 5, 6, 7, 8]"
cluster_gene_entries,90
unique_genes,78
verified_reference_rows,231
verified_unique_pmids,224


All-cluster Phase 7 extension completed and validated.
